# Create Train-Test-Validate Splits

In [7]:
import pandas as pd
from sklearn.utils import shuffle

In [77]:
def load_data(
    dir: str,
    file: str,
    dataset: str,
    primary_secondary: str = 'secondary',
    default: bool = True,
    split: list = [60, 20, 20]
    ):

    # Load Data
    df_all = pd.read_parquet(dir + file)

    # Randomise
    df = shuffle(df_all[df_all['source'] == dataset]).reset_index(drop=True)
    print(f'# of records: {len(df)}')

    # Rename mapping
    if primary_secondary == 'secondary':
        col = ['text_clean', 'secondary_anger', 'secondary_disgust', 'secondary_fear', 'secondary_joy', 'secondary_sadness', 'secondary_surprise']
        rename_col = {'secondary_anger': 'anger', 'secondary_disgust': 'disgust', 'secondary_fear': 'fear', 
                        'secondary_joy': 'joy', 'secondary_sadness': 'sadness', 'secondary_surprise': 'surprise'}
    else:
        col = ['text_clean', 'primary_anger', 'primary_disgust', 'primary_fear', 'primary_joy', 'primary_sadness', 'primary_surprise']
        rename_col = {'primary_anger': 'anger', 'primary_disgust': 'disgust', 'primary_fear': 'fear', 
                        'primary_joy': 'joy', 'primary_sadness': 'sadness', 'primary_surprise': 'surprise'}

    # Convert True/False to 1/0
    for c in rename_col.keys():
        df[c] = df[c].apply(lambda x: 1 if x == True else 0)

    if dataset in ['affectivetext', 'emoint', 'goemotions'] and default==True:
        train = df.loc[df['dataset'] == 'train', col]
        test = df.loc[df['dataset'] == 'test', col]
        val = df.loc[df['dataset'] == 'validation', col]
    elif dataset == 'ssec' and default==True:
        train = df.loc[df['dataset'] == 'train', col]
        test = df.loc[df['dataset'] == 'test', col]
        val = df.loc[df['dataset'] == 'test', col]
    else:
        splits = [int(round(sum(split[:i+1])/100 * len(df), 0)) for i,n in enumerate(split)]
        if len(df) % 3:
            splits[0] += 1
        train = df.loc[:splits[0]+1, col]
        test = df.loc[splits[0]:splits[1]+1, col]
        val = df.loc[splits[1]:, col]

    train = train.rename(columns=rename_col)
    test = test.rename(columns=rename_col)
    val = val.rename(columns=rename_col)

    train['dataset'] = 'train'
    test['dataset'] = 'test'
    val['dataset'] = 'val'
    
    print(f"Split: {'-'.join([str(i) for i in split])}")
    print(f"Train: {len(train)}")
    print(f"Test: {len(test)}")
    print(f"Validation: {len(val)}")

    return pd.concat([train, test, val])